<a href="https://colab.research.google.com/github/ajasja/prosculpt/blob/main/Prosculpt_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title setup **RFdiffusion** (~2min, run this only once)
%%time
import os, time, signal
import sys, random, string, re
if not os.path.isdir("params"):
  os.system("apt-get install aria2")
  os.system("mkdir params")
  # send param download into background
  os.system("(\
  aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C params; \
  touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  # 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies
  # 25Aug2025: updating dgi install to work with latest pytorch
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")


if not os.path.isdir("RFdiffusion/models"):
  print("downloading RFdiffusion params...")
  os.system("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt","Complex_beta_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2"):
      time.sleep(5)
  os.system(f"mv {' '.join(models)} RFdiffusion/models")
  os.system("unzip schedules.zip; rm schedules.zip")

if 'RFdiffusion' not in sys.path:
  os.environ["DGLBACKEND"] = "pytorch"
  sys.path.append('RFdiffusion')


CPU times: user 563 µs, sys: 26 µs, total: 589 µs
Wall time: 490 µs


In [ ]:
#@title setup **ProteinMPNN**
if not os.path.isdir("ProteinMPNN"):
  print("installing ProteinMPNN...")
  os.system("git clone https://github.com/dauparas/ProteinMPNN")
  #os.system("pip install pytorch torchvision torchaudio cudatoolkit=11.3 -c pytorch")

installing ProteinMPNN...


In [ ]:
#@title setup **Prosculpt**
if not os.path.isdir("prosculpt"):
  print("installing Prosculpt...")
  os.system("git clone https://github.com/ajasja/prosculpt.git")
  #os.system("pip install pytorch torchvision torchaudio cudatoolkit=11.3 -c pytorch")


os.rename("/content/prosculpt/config/installation.yaml.collab_template", "/content/prosculpt/config/installation.yaml")
os.system("pip install prosculpt/")
os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization")
os.system("pip install pymol-open-source-whl")
os.system("pip install boltz[cuda] -U")

os.system("pip install py3Dmol ipywidgets")

installing Prosculpt...


0

#Instructions

1) Fill in the options in the next cell. You can upload and use your own pdb and alignment files
2) Run the cell to generate a yaml configuration file
3) Run the next cell to execute the job
4) Download your results using the last cell

Refer to the prosculpt and rfdiffusion github pages for more information on each of the options. The default settings are an example generating a binder and can be ran as-is

https://github.com/ajasja/prosculpt/
https://github.com/RosettaCommons/RFdiffusion

In [ ]:
#@title Generate configuration file
import yaml
import os
import ast

class PlainString(str):
    pass

def plain_string_representer(dumper, data):
    return dumper.represent_scalar(
        'tag:yaml.org,2002:str',
        data,
        style=''
    )

yaml.add_representer(PlainString, plain_string_representer)

class FlowStyleList(list):
    pass

def flow_style_list_representer(dumper, data):
    return dumper.represent_sequence(
        'tag:yaml.org,2002:seq',
        data,
        flow_style=True
    )

yaml.add_representer(FlowStyleList, flow_style_list_representer)

from IPython.display import display, FileLink

# --- @param Definitions ---
# @markdown ---
# @markdown ## Basics
pdb_path = '/content/prosculpt/Examples/input_data/insulin_target.pdb' # @param {type:"string"}
output_dir = 'Test_Output' # @param {type:"string"}
contig = '[B1-150/0 70-100]' # @param {type:"string"}
num_designs_rfdiff = 2 # @param {type:"integer"}
num_seq_per_target_mpnn = 2 # @param {type:"integer"}
num_models = 1 # @param {type:"integer"}
sampling_temp = 0.1 # @param {type:"number"}
backbone_noise = 0.0 # @param {type:"number"}
model_monomer = False # @param {type:"boolean"}
symmetry = "" # @param {type:"string"}
use_backbone_filter = True # @param {type:"boolean"}
# @markdown ---
# @markdown ## Alignment
use_a3m = True # @param {type:"boolean"}
a3m_dir = "/content/prosculpt/Examples/input_data/Insulin_Alignment" # @param {type:"string"} # Used only if 'use_a3m' is True
prediction_model = "Boltz2"
# @markdown ---
# @markdown ## Partial diffusion
partial_diffusion = False # @param {type:"boolean"}
partial_T = 5 # @param {type:"integer"}

provide_seq= "" #@param {type:"string"}

inference_section={'generatedWithProsculpt':True}
potentials_section={'generatedWithProsculpt':True}
ppi_section={}
diffuser_section={}
diffuser_section['partial_T'] = partial_T
denoiser_section={}
contigmap_section={}

# @markdown ---
# @markdown ## Inpainting
inpaint = False # @param {type:"boolean"}
inpaint_seq= "" #@param {type:"string"}


if symmetry != "":
  inference_section['symmetry']=symmetry
# @markdown ---
# @markdown ## Hotspots
hotspot_res = "['A59','A83','A91']" #@param{type:"string"}

included_sections=['inference','potentials']
if inpaint_seq != "" and inpaint==True:
  contigmap_section['inpaint_seq']=inpaint_seq
  included_sections.append('contigmap')

if provide_seq != "" and partial_diffusion==True:
  contigmap_section['provide_seq']=FlowStyleList(
    [PlainString(x) for x in ast.literal_eval(provide_seq)]
)
  if 'contigmap' not in included_sections:
    included_sections.append('contigmap')

if hotspot_res != "":
  ppi_section['hotspot_res'] = FlowStyleList([PlainString(x) for x in ast.literal_eval(hotspot_res)])
  included_sections.append('ppi')



# --- Generate YAML Functionality ---
def generate_yaml_file():
    config = {
        'pdb_path': pdb_path,
        'output_dir': output_dir,
        'contig': contig,
        'num_designs_rfdiff': num_designs_rfdiff,
        'num_seq_per_target_mpnn': num_seq_per_target_mpnn,
        'af2_mpnn_cycles': 1,
        'num_models': num_models,
        'chain_break_cutoff_A': 2,
        'sampling_temp': sampling_temp,
        'backbone_noise': backbone_noise,
        'use_a3m': use_a3m,
        'model_monomer': model_monomer,
        'partial_diffusion': partial_diffusion,
        'prediction_model': prediction_model,
        'defaults': ['installation', '_self_'],
        'pass_to_rfdiff': included_sections,
        'contigmap': contigmap_section,
        'inference': inference_section,
        'potentials': potentials_section,
        'ppi': ppi_section,
        'diffuser': diffuser_section,
        'denoiser': denoiser_section
    }



    # Add a3m_dir only if use_a3m is True
    if use_a3m:
        config['a3m_dir'] = a3m_dir

    if use_backbone_filter:
      config['rfdiff_backbone_filters'] = [{'filter_name': 'SS_filter','filter_script':'/content/prosculpt/plugins/SS_filter.py', 'delete_failed':False}]

    yaml_filename = 'generated_config_file.yaml'
    with open(yaml_filename, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

    print(f"YAML file '{yaml_filename}' generated successfully!")
    #print("--- Content of config.yaml ---")
    #print(yaml.dump(config, default_flow_style=False, sort_keys=False))
    #display(FileLink(yaml_filename))

# Call the function to generate the YAML file
generate_yaml_file()

YAML file 'generated_config_file.yaml' generated successfully!


In [ ]:
# @title Run generated config (10-15 minutes for the example)
import subprocess
import sys

cmd = [
    'python',
    '/content/prosculpt/rfdiff_mpnn_af2_merged.py',
    '-cd',
    '.',
    '-cn',
    'generated_config_file'
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

return_code = process.wait()

if return_code != 0:
    print(f"\nCommand failed with exit code {return_code}")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu.cxx11thread.serialization 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org
File:  /content/prosculpt/rfdiff_mpnn_af2_merged.py
[2026-05-27 14:54:34,070][__main__][INFO] - Hydrić
[2026-05-27 14:54:34,072]

In [ ]:
# @title Display results

import os
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output

# === CONFIG ===
PDB_DIR = f"/content/{output_dir}/final_pdbs"  # <-- change this to your directory

# Get all PDB files
pdb_files = sorted([
    f for f in os.listdir(PDB_DIR)
    if f.lower().endswith(".pdb")
])

if not pdb_files:
    raise ValueError(f"No PDB files found in: {PDB_DIR}")

# Dropdown selector
pdb_selector = widgets.Dropdown(
    options=pdb_files,
    description="Select pdb:",
    layout=widgets.Layout(width='600px')
)

# Display button
show_button = widgets.Button(
    description="Show Protein",
    button_style='primary'
)

# Output area
output = widgets.Output()

def show_pdb(_):
    with output:
        clear_output(wait=True)

        pdb_path = os.path.join(PDB_DIR, pdb_selector.value)

        with open(pdb_path, "r") as f:
            pdb_str = f.read()

        hbondCutoff = 4.0

        view = py3Dmol.view(
            width=800,
            height=600,
            js='https://3dmol.org/build/3Dmol.js'
        )

        view.addModel(
            pdb_str,
            'pdb',
            {'hbondCutoff': hbondCutoff}
        )

        # Cartoon colored by B-factor
        view.setStyle({
            'cartoon': {
                'colorscheme': {
                    'prop': 'b',
                    'gradient': 'roygb',
                    'min': 0,
                    'max': 100
                }
            }
        })

        # Add sticks
        view.addStyle({}, {
            'stick': {
                'colorscheme': "GrayCarbon"
            }
        })

        view.zoomTo()
        view.show()

show_button.on_click(show_pdb)

display(widgets.VBox([
    pdb_selector,
    show_button,
    output
]))

In [ ]:
#@title Download results

import shutil
from google.colab import files

folder_path = f"/content/{output_dir}"
output_path = "/content/output.zip"

shutil.make_archive(output_path.replace(".zip", ""), 'zip', folder_path)

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>